# Component ablation on MCPEvol-Bench (60 cases) - `mcp-hallucination-checker`

What does each component of the checker contribute? Every variant is evaluated on the same 60 cases in
`benchmarks/mcpevol.json` (29 consistent / 31 hallucinated; types: content, format, schema, negation, role_swap),
using the repository's own code (imported, not re-implemented), except variant 5 which is an experimental override defined in this notebook.

| # | Variant | Definition |
|---|---------|-----------|
| 1 | **schema-only** | `run_schema_alignment` for `schema`/`format` cases. Text-only cases (content/negation/role_swap) have no schema, so this variant **predicts "consistent"** for them. |
| 2 | **NLI-only** | whole hypothesis vs premise entailment (`entailment.score_spans(premise, [hypothesis])`, windowed, repo's hypothesis template), `consistent` iff score >= entailment threshold. Applied to all 60 cases using their `hypothesis`/`premise` text. (2b: same with the raw hypothesis, no repo template.) |
| 3 | **GED-only** | `compute_ged` normalised distance <= `ged_threshold`, no NLI span gating. Applied to all 60 cases using their text. |
| 4 | **hybrid (repo)** | the benchmark runner's routing: `schema`/`format` -> `run_schema_alignment`; others -> `run_verification` (GED **and** per-span NLI gate). Cross-checked against the real `_run_record` at the fixed config. |
| 5 | **hybrid + asymmetric GED (EXPERIMENTAL, defined in this notebook)** | same routing as 4, but GED is hypothesis-in-premise: no canonical swap, normalised by hypothesis size only, only hypothesis-side unmatched value nodes are counted. Implemented by monkeypatching `server.tools.verify_logic.compute_ged`; `server/pipeline/ged.py` is **not** modified. |

Also reported: component *leave-one-out* ablations of the hybrid, threshold grids (fixed config vs best-on-this-set vs leave-one-out CV), per-type metrics,
bootstrap 95% CIs, confusion matrices and pairwise exact McNemar tests.

**Warnings**
* **n = 60** (and only 2-5 cases per class for negation / role_swap). All confidence intervals are very wide; differences of a few cases are not significant (see McNemar).
* "Best-on-this-set" thresholds are tuned **on the same 60 cases they are scored on** and are therefore optimistic. The leave-one-out column is a less biased estimate, but the benchmark itself is small and synthetic and was also used to develop the pipeline.
* Positive class for P/R/F1 is **hallucinated** (what a detector should find). The repo runner uses "consistent" as positive; its F1 is reported as `f1_cons`.

**Setup:** Kaggle -> *Add Input* -> the dataset made from `package_for_kaggle.py` (zip is auto-detected). Settings: **Internet = On** (downloads `dslim/bert-base-NER`, `cross-encoder/nli-deberta-v3-base` and the spaCy model once); GPU optional (only helps the NLI/NER pipelines, and Part 2).

## 1. Setup: dependencies, repo package, spaCy model

In [ ]:
import os, sys, subprocess, importlib, glob, zipfile, json, time, math, re, random, contextlib, threading, warnings, itertools, platform, functools
from pathlib import Path
warnings.filterwarnings("ignore")

def _ensure(pip_name, import_name=None):
    try:
        importlib.import_module(import_name or pip_name)
    except ImportError:
        print("pip install", pip_name)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

# Minimum for the modules this notebook imports (fastmcp is only needed by server/main.py, NOT imported here).
# Kaggle images already ship most of these; nothing is upgraded.
for _pip, _imp in [("numpy", None), ("pandas", None), ("scipy", None), ("networkx", None), ("pydantic", None),
                   ("pyyaml", "yaml"), ("spacy", None), ("torch", None), ("transformers", None),
                   ("sentencepiece", None), ("protobuf", "google.protobuf"), ("tqdm", None)]:
    _ensure(_pip, _imp)

import numpy as np, pandas as pd, torch, transformers, spacy
print("python", platform.python_version(), "| torch", torch.__version__, "| transformers", transformers.__version__, "| spacy", spacy.__version__)

# ---- locate the repo package (dataset made by kaggle/package_for_kaggle.py) -------------------------
def _has_pkg(d):
    return (Path(d) / "server" / "pipeline" / "ged.py").exists()

def locate_package():
    env = os.environ.get("ABL_PKG_DIR")
    if env and _has_pkg(env):
        return Path(env)
    # 1) Kaggle auto-extracts uploaded zips -> a folder containing server/pipeline/ged.py
    for ged in sorted(glob.glob("/kaggle/input/**/server/pipeline/ged.py", recursive=True)):
        return Path(ged).parents[2]
    # 2) a raw zip somewhere under /kaggle/input
    for z in sorted(glob.glob("/kaggle/input/**/*.zip", recursive=True)):
        try:
            with zipfile.ZipFile(z) as zf:
                if any(n.endswith("server/pipeline/ged.py") for n in zf.namelist()):
                    dest = Path("/kaggle/working/pkg")
                    dest.mkdir(parents=True, exist_ok=True)
                    zf.extractall(dest)
                    for ged in dest.rglob("server/pipeline/ged.py"):
                        return ged.parents[2]
        except zipfile.BadZipFile:
            pass
    # 3) local run: walk up from the cwd
    for d in [Path.cwd(), *Path.cwd().parents]:
        if _has_pkg(d):
            return d
    raise FileNotFoundError("Repo package not found. Upload the zip from kaggle/package_for_kaggle.py as a Kaggle "
                            "dataset and attach it to this notebook (Add Input).")

PKG_ROOT = locate_package()
if str(PKG_ROOT) not in sys.path:
    sys.path.insert(0, str(PKG_ROOT))
print("package root:", PKG_ROOT)

# ---- spaCy model: the repo loads en_core_web_trf and falls back to en_core_web_sm --------------------
USE_TRF = True            # True: install/use en_core_web_trf (the repo's preferred model, ~450 MB download, slower). False: en_core_web_sm only.
                          # NOTE: results can differ between trf and sm (different NER/parses -> different graphs). The model used is recorded in the results JSON.
SPACY_MODELS = ["en_core_web_trf", "en_core_web_sm"] if USE_TRF else ["en_core_web_sm"]   # same preference order as server/pipeline/srl.py
INSTALL_SPACY_MODEL = True

def _try_load(name):
    try:
        spacy.load(name)
        return True
    except Exception:
        return False

if not any(_try_load(m) for m in SPACY_MODELS) and INSTALL_SPACY_MODEL:
    for m in SPACY_MODELS:
        print("installing spaCy model", m, "(needs Internet = On)")
        r = subprocess.run([sys.executable, "-m", "spacy", "download", m], capture_output=True, text=True)
        importlib.invalidate_caches()
        if _try_load(m):
            break
        print(r.stdout[-400:], r.stderr[-400:])
USED_SPACY = next((m for m in SPACY_MODELS if _try_load(m)), None)
assert USED_SPACY, "No spaCy model could be loaded. Turn Internet on, or restart the kernel after install and re-run."
print("spaCy model available (repo prefers trf, falls back to sm):", USED_SPACY)

## 2. Configuration

In [ ]:
SEED = 42
N_BOOT = 2000                        # bootstrap resamples (n=60 -> intervals are wide)
ABLATE_AGGREFACT_SAMPLE = False      # Part 2 (optional, slow): GED+NLI vs NLI-only on a stratified AggreFact sample
USE_GPU_FOR_HF = True                # move the repo's NLI + BERT-NER HF pipelines to GPU if available (does not change the logic)

# smoke-test knob (used for local verification only): keep the first K cases per (type, label)
SMOKE_PER_GROUP = int(os.environ["ABL_SMOKE"]) if os.environ.get("ABL_SMOKE") else None

# threshold grids for the sweep
GED_GRID = [round(x, 2) for x in np.arange(0.05, 0.851, 0.05)]
ENT_GRID = [round(x, 2) for x in np.arange(0.05, 0.951, 0.05)]

# Part 2 settings
AGG_N = 300
AGG_SEED = 42
AGG_GRAPH_DOC_WORDS = 250            # docs are truncated to this many words for graph building (GED); NLI uses the full doc, windowed
AGG_TIMEOUT_S = 90                   # per-record wall-clock limit for graph building + GED
AGG_DIR = os.environ.get("ABL_AGG_DIR")   # optional: folder containing test-*.parquet (else auto-detected under /kaggle/input)

OUT_DIR = Path(os.environ.get("ABL_OUT_DIR") or ("/kaggle/working" if os.path.isdir("/kaggle/working") else "./ablation_out"))
OUT_DIR.mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("output dir:", OUT_DIR)

## 3. Import the repo's own components
Everything below is imported from the package. The only things defined here are (a) metric/statistics helpers and (b) the experimental asymmetric GED (variant 5).

In [ ]:
from server import config as repo_config
from server.pipeline import entailment, srl, ged as ged_mod
from server.pipeline.graph_builder import build_semantic_graph
from server.pipeline.ged import compute_ged, _node_subst_cost, _edge_subst_cost, _is_value_node
from server.pipeline.srl import extract_frames
from server.tools import verify_logic
from server.tools.verify_logic import run_verification, _extract_argument_spans
from server.tools.schema import run_schema_alignment
from benchmarks.mcpevol_runner import _run_record          # the benchmark runner's actual routing

GED_THR = repo_config.GED_THRESHOLD                 # fixed config from configs/thresholds.yaml
ENT_THR = repo_config.ENTAILMENT_THRESHOLD
FAST_LIMIT = repo_config.GED_FAST_MODE_NODE_LIMIT
assert verify_logic.ENTAILMENT_THRESHOLD == ENT_THR
print(f"fixed config: ged_threshold={GED_THR}  entailment_threshold={ENT_THR}  ged_fast_mode_node_limit={FAST_LIMIT}")

# Optional: run the repo's two HF pipelines on GPU (identical models/logic, only the device changes).
if USE_GPU_FOR_HF and torch.cuda.is_available():
    from transformers import pipeline as _hf_pipeline
    entailment._load_pipeline.cache_clear()
    entailment._load_pipeline = functools.lru_cache(maxsize=1)(lambda: _hf_pipeline(
        "text-classification", model=entailment.MODEL_NAME, top_k=None, truncation=True, device=0))
    srl._load_ner.cache_clear()
    srl._load_ner = functools.lru_cache(maxsize=1)(lambda: _hf_pipeline(
        "token-classification", model="dslim/bert-base-NER", aggregation_strategy="simple", device=0))
    print("HF pipelines (NLI, NER) -> GPU")
else:
    print("HF pipelines on CPU (fine for 60 cases)")

# which spaCy model does the repo actually use at runtime?
_nlp_meta = srl._load_nlp().meta
SPACY_USED = f"{_nlp_meta['lang']}_{_nlp_meta['name']}"
print("repo's spaCy model at runtime:", SPACY_USED, _nlp_meta.get("version"))

## 4. Experimental variant 5: asymmetric (hypothesis-in-premise) GED
**EXPERIMENTAL - not part of the repository.** Differences from `server/pipeline/ged.compute_ged`:
* no canonical swap (hypothesis stays the "source" graph), so the score is *not* symmetric;
* premise-only nodes/edges are free (the premise may legitimately say more than the hypothesis);
* normalised by hypothesis size only: `raw / (|V_hyp| + |E_hyp|)`;
* value-node bonus (+0.20 each) counts only hypothesis value nodes that are absent from the premise.

It reuses the repo's node/edge substitution costs (`_node_subst_cost`, `_edge_subst_cost`, `_is_value_node`) and the same fast-mode node limit
(in fast mode a simple exact-ID approximation is used: hypothesis nodes/edges not present verbatim in the premise).
The threshold scale differs from the symmetric GED, so 0.35 is not necessarily a sensible value for it.

In [ ]:
from scipy.optimize import linear_sum_assignment

VALUE_MISMATCH_WEIGHT = 0.20   # same as ged.compute_ged

def compute_ged_asymmetric(G_hyp, G_prem):
    """EXPERIMENTAL hypothesis-in-premise GED. Returns (raw, normalised in [0,1])."""
    nh, npm = G_hyp.number_of_nodes(), G_prem.number_of_nodes()
    denom = nh + G_hyp.number_of_edges()
    if denom == 0:
        return 0.0, 0.0
    hn, pn = list(G_hyp.nodes(data=True)), list(G_prem.nodes(data=True))
    if nh + npm > FAST_LIMIT:                       # fast-mode approximation (exact-ID containment)
        pids = set(G_prem.nodes)
        pedges = set(G_prem.edges)
        raw = float(sum(1 for n, _ in hn if n not in pids) + sum(1 for e in G_hyp.edges if e not in pedges))
    elif npm == 0:
        raw = float(nh + G_hyp.number_of_edges())
    else:
        C = np.array([[_node_subst_cost(a, aa, b, ba) for b, ba in pn] for a, aa in hn])
        rows, cols = linear_sum_assignment(C)       # rectangular: matches min(nh, npm) pairs
        raw = float(C[rows, cols].sum()) + 1.0 * (nh - len(rows))   # unmatched hypothesis nodes cost 1.0; unmatched premise nodes are free
        nmap = {hn[i][0]: pn[j][0] for i, j in zip(rows, cols)}
        for u, v, d in G_hyp.edges(data=True):
            if u in nmap and v in nmap:
                mu, mv = nmap[u], nmap[v]
                raw += _edge_subst_cost(d, G_prem[mu][mv]) if G_prem.has_edge(mu, mv) else 1.0
    norm = max(0.0, min(1.0, raw / denom))
    hv = {n for n, a in hn if _is_value_node(n, a)}
    pv = {n for n, a in pn if _is_value_node(n, a)}
    norm = min(1.0, norm + VALUE_MISMATCH_WEIGHT * len(hv - pv))
    return raw, norm

@contextlib.contextmanager
def asymmetric_ged_patch():
    """Temporarily replace the GED used inside run_verification (repo file untouched)."""
    orig = verify_logic.compute_ged
    verify_logic.compute_ged = compute_ged_asymmetric
    try:
        yield
    finally:
        verify_logic.compute_ged = orig

# tiny sanity checks (graph-level, no models needed)
import networkx as nx
_g1 = nx.DiGraph(); _g1.add_node("pay", kind="predicate", negated=False); _g1.add_node("$50", kind="MONEY"); _g1.add_edge("pay", "$50", label="MONEY")
_g2 = _g1.copy(); _g2.add_node("Bob", kind="argument"); _g2.add_edge("pay", "Bob", label="PER")
print("asym(hyp=small, prem=small+extra) =", compute_ged_asymmetric(_g1, _g2)[1], "(premise extras are free)")
print("asym(hyp=small+extra, prem=small) =", compute_ged_asymmetric(_g2, _g1)[1], "(hypothesis extras are penalised)")
print("repo symmetric both ways          =", compute_ged(_g1, _g2)[1], compute_ged(_g2, _g1)[1])

## 5. Load the benchmark and compute per-case features once
All variants and all threshold grids are derived from these cached scores, so the sweep does not re-run any model.

In [ ]:
DATA = json.loads((PKG_ROOT / "benchmarks" / "mcpevol.json").read_text(encoding="utf-8"))
if SMOKE_PER_GROUP:
    seen, keep = {}, []
    for r in DATA:
        k = (r["type"], r["label"]); seen[k] = seen.get(k, 0) + 1
        if seen[k] <= SMOKE_PER_GROUP: keep.append(r)
    DATA = keep
    print(f"SMOKE MODE: {len(DATA)} cases")
TYPE_ORDER = ["content", "negation", "role_swap", "format", "schema"]
TYPED_TYPES = ("schema", "format")      # routed to run_schema_alignment by the runner
print(len(DATA), "cases | consistent:", sum(r["label"] for r in DATA), "| hallucinated:", sum(not r["label"] for r in DATA))
print(pd.crosstab(pd.Series([r["type"] for r in DATA], name="type"), pd.Series([r["label"] for r in DATA], name="label (True=consistent)")))

def raw_nli(premise, hypothesis):
    """Whole-hypothesis entailment WITHOUT the repo's hypothesis template (variant 2b); same windowing + max-over-windows."""
    clf = entailment._load_pipeline()
    out = clf([{"text": w, "text_pair": hypothesis} for w in entailment._premise_windows(premise)])
    best = 0.0
    for scores in out:
        if isinstance(scores, list) and scores and isinstance(scores[0], list):
            scores = scores[0]
        best = max(best, next((x["score"] for x in scores if x["label"].lower() == "entailment"), 0.0))
    return best

def case_features(rec):
    h, p = rec["hypothesis"], rec["premise"]
    Gh, Gp = build_semantic_graph(h), build_semantic_graph(p)
    _, g_sym = compute_ged(Gh, Gp)
    _, g_asym = compute_ged_asymmetric(Gh, Gp)
    spans = _extract_argument_spans(extract_frames(h))          # exactly what run_verification sends to NLI
    sp = entailment.score_spans(p, spans) if spans else []
    min_span = min((s for _, s in sp), default=1.0)             # no spans -> nothing can be 'low' -> gate passes
    f = dict(id=rec["id"], type=rec["type"], label=bool(rec["label"]),
             ged_sym=g_sym, ged_asym=g_asym, min_span=min_span, n_spans=len(spans),
             nli_templ=entailment.score_spans(p, [h])[0][1] if h.strip() else 0.0,
             nli_raw=raw_nli(p, h) if h.strip() else 0.0,
             hyp_nodes=Gh.number_of_nodes(), prem_nodes=Gp.number_of_nodes(),
             fast_mode=(Gh.number_of_nodes() + Gp.number_of_nodes()) > FAST_LIMIT)
    if rec["type"] in TYPED_TYPES:
        aa = rec.get("agent_args", {}); ts = rec.get("tool_schema")
        res = run_schema_alignment(aa, ts)
        f["schema_aligned"] = bool(res.is_aligned)
    else:
        f["schema_aligned"] = np.nan
    return f

t0 = time.time()
FEAT = pd.DataFrame([case_features(r) for r in DATA])
print(f"features computed in {time.time()-t0:.1f}s")
Y = FEAT["label"].values.astype(bool)                # True = consistent
TYPED = FEAT["type"].isin(TYPED_TYPES).values
TYPES = FEAT["type"].values
print(FEAT.drop(columns=["label"]).round(3).to_string())

## 6. Variant definitions and fidelity check
Predictions are `True` = "consistent". The hybrid composition used for the threshold grid is
`typed -> schema_aligned`, `text -> (ged <= ged_threshold) and (no span with entailment < entailment_threshold)`, which is exactly what
`run_verification` + the runner do; below it is **verified against the real code** at the fixed config.

In [ ]:
SCHEMA_ALIGNED = np.where(TYPED, FEAT["schema_aligned"].fillna(True).astype(bool).values, True)

def predict(variant, g=None, e=None):
    g = GED_THR if g is None else g
    e = ENT_THR if e is None else e
    ged_ok = lambda col: FEAT[col].values <= g
    gate_ok = FEAT["min_span"].values >= e
    if variant == "1 schema-only":          return SCHEMA_ALIGNED.copy()                         # text-only cases -> 'consistent'
    if variant == "2 NLI-only":             return FEAT["nli_templ"].values >= e
    if variant == "2b NLI-only (raw hyp)":  return FEAT["nli_raw"].values >= e
    if variant == "3 GED-only":             return ged_ok("ged_sym")
    if variant == "4 hybrid (repo)":        return np.where(TYPED, SCHEMA_ALIGNED, ged_ok("ged_sym") & gate_ok)
    if variant == "5 hybrid + asym GED [EXPERIMENTAL]":
                                            return np.where(TYPED, SCHEMA_ALIGNED, ged_ok("ged_asym") & gate_ok)
    # component leave-one-out ablations of the hybrid
    if variant == "4-GED (schema + NLI span gate)":   return np.where(TYPED, SCHEMA_ALIGNED, gate_ok)
    if variant == "4-NLI (schema + GED)":             return np.where(TYPED, SCHEMA_ALIGNED, ged_ok("ged_sym"))
    if variant == "4-schema (text pipeline on all 60)": return ged_ok("ged_sym") & gate_ok
    raise KeyError(variant)

MAIN = ["1 schema-only", "2 NLI-only", "3 GED-only", "4 hybrid (repo)", "5 hybrid + asym GED [EXPERIMENTAL]"]
EXTRA = ["2b NLI-only (raw hyp)"]
LOO_COMP = ["4-GED (schema + NLI span gate)", "4-NLI (schema + GED)", "4-schema (text pipeline on all 60)"]
# which parameters each variant actually depends on
PARAMS = {"1 schema-only": (), "2 NLI-only": ("e",), "2b NLI-only (raw hyp)": ("e",), "3 GED-only": ("g",),
          "4 hybrid (repo)": ("g", "e"), "5 hybrid + asym GED [EXPERIMENTAL]": ("g", "e"),
          "4-GED (schema + NLI span gate)": ("e",), "4-NLI (schema + GED)": ("g",), "4-schema (text pipeline on all 60)": ("g", "e")}

# ---- fidelity: composed hybrid vs the REAL runner (_run_record) and real run_verification ---------------------------
real = np.array([_run_record(r, GED_THR)[0] for r in DATA], dtype=bool)
composed = predict("4 hybrid (repo)")
print(f"hybrid (composed from cache) vs real runner _run_record: {int((real != composed).sum())} disagreements / {len(DATA)}")
if (real != composed).any():
    print("  differing ids:", FEAT.id[real != composed].tolist())

with asymmetric_ged_patch():
    real5 = np.array([_run_record(r, GED_THR)[0] for r in DATA], dtype=bool)
comp5 = predict("5 hybrid + asym GED [EXPERIMENTAL]")
print(f"variant 5 (composed) vs run_verification with monkeypatched GED: {int((real5 != comp5).sum())} disagreements / {len(DATA)}")
if (real5 != comp5).any():
    print("  differing ids:", FEAT.id[real5 != comp5].tolist())
assert verify_logic.compute_ged is compute_ged, "monkeypatch was not restored"

## 7. Metrics, bootstrap CIs, McNemar

In [ ]:
def _div(a, b): return a / b if b else 0.0

def metrics(y, p):
    """y,p: bool arrays, True = consistent. P/R/F1 use HALLUCINATED as the positive class; f1_cons uses the runner's convention."""
    y, p = np.asarray(y, bool), np.asarray(p, bool)
    cc, ch = int((y & p).sum()), int((y & ~p).sum())            # consistent predicted consistent / flagged
    hc, hh = int((~y & p).sum()), int((~y & ~p).sum())          # hallucinated missed / caught
    n = len(y)
    prec_h, rec_h = _div(hh, hh + ch), _div(hh, hh + hc)
    prec_c, rec_c = _div(cc, cc + hc), _div(cc, cc + ch)
    f = lambda P, R: _div(2 * P * R, P + R)
    return dict(n=n, acc=_div(cc + hh, n), bal_acc=0.5 * (rec_h + rec_c), prec_h=prec_h, rec_h=rec_h, f1_h=f(prec_h, rec_h),
                f1_cons=f(prec_c, rec_c), cons_as_cons=cc, cons_as_hall=ch, hall_as_cons=hc, hall_as_hall=hh)

def bootstrap_ci(y, p, B=N_BOOT, seed=SEED):
    y, p = np.asarray(y, bool), np.asarray(p, bool); n = len(y)
    idx = np.random.default_rng(seed).integers(0, n, (B, n))
    yy, pp = y[idx], p[idx]
    hh = (~yy & ~pp).sum(1); ch = (yy & ~pp).sum(1); hc = (~yy & pp).sum(1); cc = (yy & pp).sum(1)
    acc = (cc + hh) / n
    P = np.divide(hh, hh + ch, out=np.zeros(B), where=(hh + ch) > 0); R = np.divide(hh, hh + hc, out=np.zeros(B), where=(hh + hc) > 0)
    F = np.divide(2 * P * R, P + R, out=np.zeros(B), where=(P + R) > 0)
    q = lambda a: (round(float(np.percentile(a, 2.5)), 3), round(float(np.percentile(a, 97.5)), 3))
    return dict(acc_ci=q(acc), f1_h_ci=q(F))

def mcnemar_exact(c1, c2):
    """Exact two-sided McNemar on per-case correctness. Returns (b, c, p): b = only A right, c = only B right."""
    c1, c2 = np.asarray(c1, bool), np.asarray(c2, bool)
    b, c = int((c1 & ~c2).sum()), int((~c1 & c2).sum()); n = b + c
    if n == 0: return b, c, 1.0
    k = min(b, c)
    return b, c, min(1.0, 2 * sum(math.comb(n, i) for i in range(k + 1)) / 2 ** n)

def fmt_ci(ci): return f"[{ci[0]:.2f}, {ci[1]:.2f}]"

## 8. Fixed-config results
Thresholds from `configs/thresholds.yaml` (not tuned on this data by this notebook; the repo's own threshold-sweep tooling may have used these 60 cases, so even "fixed" is not independent).

In [ ]:
def summary_table(variants, g=None, e=None):
    rows = []
    for v in variants:
        p = predict(v, g, e); m = metrics(Y, p); ci = bootstrap_ci(Y, p)
        rows.append({"variant": v, "acc": m["acc"], "acc 95% CI": fmt_ci(ci["acc_ci"]), "bal_acc": m["bal_acc"],
                     "P(hall)": m["prec_h"], "R(hall)": m["rec_h"], "F1(hall)": m["f1_h"], "F1(hall) 95% CI": fmt_ci(ci["f1_h_ci"]),
                     "f1_cons": m["f1_cons"]})
    return pd.DataFrame(rows).set_index("variant")

T_FIXED = summary_table(MAIN + EXTRA + LOO_COMP)
print(f"FIXED CONFIG  ged_threshold={GED_THR}  entailment_threshold={ENT_THR}   (n={len(Y)}, bootstrap B={N_BOOT})")
print("Majority-class baseline (always 'consistent'):", round(Y.mean(), 3), "| always 'hallucinated':", round(1 - Y.mean(), 3))
print(T_FIXED.round(3).to_string())

print("\nConfusion matrices (rows = truth, cols = prediction)")
for v in MAIN + EXTRA:
    m = metrics(Y, predict(v))
    print(f"\n{v}\n{'':>22}pred consistent  pred hallucinated\n{'true consistent':>22}{m['cons_as_cons']:>15}{m['cons_as_hall']:>18}\n{'true hallucinated':>22}{m['hall_as_cons']:>15}{m['hall_as_hall']:>18}")

### Per-type metrics at the fixed config (positive class = hallucinated; negation/role_swap have only 5 cases each - read as anecdotes)

In [ ]:
def per_type_table(variants, g=None, e=None):
    rows = []
    for v in variants:
        p = predict(v, g, e)
        for t in TYPE_ORDER + ["ALL text-routed (content/negation/role_swap)", "ALL typed-routed (schema/format)"]:
            if t in TYPE_ORDER: msk = TYPES == t
            elif t.startswith("ALL text"): msk = ~TYPED
            else: msk = TYPED
            if not msk.any(): continue
            m = metrics(Y[msk], p[msk])
            rows.append({"variant": v, "type": t, "n": m["n"], "n_cons/hall": f"{int(Y[msk].sum())}/{int((~Y[msk]).sum())}",
                         "acc": m["acc"], "P(hall)": m["prec_h"], "R(hall)": m["rec_h"], "F1(hall)": m["f1_h"]})
    return pd.DataFrame(rows)

PT_FIXED = per_type_table(MAIN)
print(PT_FIXED.round(3).to_string(index=False))
print("\nAccuracy pivot (variant x type):")
print(PT_FIXED.pivot(index="variant", columns="type", values="acc").round(2).to_string())

## 9. Threshold sweeps: fixed config vs best-on-this-set vs leave-one-out
**WARNING - optimistic.** "Best-on-this-set" picks the threshold(s) maximising accuracy on the very 60 cases it is then scored on.
`LOO-CV acc` re-selects the thresholds on 59 cases and predicts the held-out one (nested leave-one-out); it is a less biased estimate.
Ties for best are broken by balanced accuracy, then by closeness to the fixed config. The number of grid points tied at the maximum shows how flat the optimum is.

In [ ]:
def grid_for(v):
    ps = PARAMS[v]
    if ps == (): return [dict()]
    if ps == ("e",): return [dict(e=e) for e in ENT_GRID]
    if ps == ("g",): return [dict(g=g) for g in GED_GRID]
    return [dict(g=g, e=e) for g in GED_GRID for e in ENT_GRID]

def sweep(v):
    grid = grid_for(v)
    P = np.array([predict(v, **kw) for kw in grid])                    # K x n
    corr = (P == Y)                                                     # K x n
    acc = corr.mean(1)
    bal = np.array([metrics(Y, p)["bal_acc"] for p in P])
    dist = np.array([abs(kw.get("g", GED_THR) - GED_THR) + abs(kw.get("e", ENT_THR) - ENT_THR) for kw in grid])
    key = acc * 1e6 + bal * 1e3 - dist                                  # lexicographic-ish tie-break
    k = int(np.argmax(key))
    n_tied = int((acc >= acc[k] - 1e-12).sum())
    # nested leave-one-out
    tot = corr.sum(1); loo_pred = np.zeros(len(Y), bool)
    for i in range(len(Y)):
        tr_acc = (tot - corr[:, i]) / (len(Y) - 1)
        kk = int(np.argmax(tr_acc * 1e6 - dist))
        loo_pred[i] = P[kk, i]
    return dict(grid=grid, P=P, acc=acc, best=k, best_params=grid[k], best_acc=float(acc[k]), n_tied=n_tied,
                n_grid=len(grid), loo_pred=loo_pred)

SWEEPS = {v: sweep(v) for v in MAIN + EXTRA + LOO_COMP}
rows = []
for v, s in SWEEPS.items():
    fixed = metrics(Y, predict(v)); best_p = s["P"][s["best"]]; bm = metrics(Y, best_p); lm = metrics(Y, s["loo_pred"])
    rows.append({"variant": v, "fixed params": f"g={GED_THR}, e={ENT_THR}" if PARAMS[v] else "-",
                 "fixed acc": fixed["acc"], "fixed F1(hall)": fixed["f1_h"],
                 "BEST-on-set params (OPTIMISTIC)": ", ".join(f"{k}={x}" for k, x in s["best_params"].items()) or "-",
                 "best acc": bm["acc"], "best F1(hall)": bm["f1_h"], "grid pts tied": f"{s['n_tied']}/{s['n_grid']}",
                 "LOO-CV acc": lm["acc"], "LOO-CV F1(hall)": lm["f1_h"]})
T_SWEEP = pd.DataFrame(rows).set_index("variant")
print("g = GED threshold (consistent iff normalised GED <= g), e = entailment threshold (consistent iff score >= e)")
print(T_SWEEP.round(3).to_string())
print("\nThe asymmetric-GED variant is on a different distance scale, so its 'fixed' row (g=0.35) is not a fair operating point; compare best/LOO columns with that in mind.")

In [ ]:
# accuracy heat-maps over the (ged, entailment) grid for the two hybrids (rows = ged threshold, cols = entailment threshold)
def acc_grid(v):
    s = SWEEPS[v]; A = np.array([s["acc"][i] for i in range(len(s["grid"]))]).reshape(len(GED_GRID), len(ENT_GRID))
    return pd.DataFrame(A, index=[f"g={g}" for g in GED_GRID], columns=[f"e={e}" for e in ENT_GRID])
for v in ["4 hybrid (repo)", "5 hybrid + asym GED [EXPERIMENTAL]"]:
    print(f"\nAccuracy grid: {v}   (optimistic: scored on the tuning set)")
    print(acc_grid(v).round(2).to_string())
try:
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    fig, axs = plt.subplots(1, 2, figsize=(14, 5))
    for ax, v in zip(axs, ["4 hybrid (repo)", "5 hybrid + asym GED [EXPERIMENTAL]"]):
        A = acc_grid(v).values; im = ax.imshow(A, vmin=0.4, vmax=1.0, aspect="auto", origin="lower", cmap="viridis")
        ax.set_xticks(range(len(ENT_GRID))[::2]); ax.set_xticklabels(ENT_GRID[::2]); ax.set_yticks(range(len(GED_GRID))[::2]); ax.set_yticklabels(GED_GRID[::2])
        ax.set_xlabel("entailment threshold"); ax.set_ylabel("GED threshold"); ax.set_title(v + "\naccuracy (tuned-on-same-60: optimistic)", fontsize=9)
        fig.colorbar(im, ax=ax)
    fig.tight_layout(); fig.savefig(OUT_DIR / "ablation_hybrid_grid.png", dpi=110); plt.show()
except Exception as ex:
    print("heat-map skipped:", ex)

## 10. Pairwise McNemar exact tests (fixed config, and best-on-set)
Per-case correctness is compared between two variants; `only A` / `only B` count the cases that exactly one variant gets right. With n=60 and a handful of discordant cases, **most pairs cannot be separated statistically**. p-values are two-sided exact and **not** corrected for the 10 comparisons.

In [ ]:
def mcnemar_matrix(preds):
    names = list(preds); rows = []
    for a, b in itertools.combinations(names, 2):
        nb, nc, p = mcnemar_exact(preds[a] == Y, preds[b] == Y)
        rows.append({"A": a, "B": b, "only A right": nb, "only B right": nc, "p (exact)": round(p, 4)})
    return pd.DataFrame(rows)

print("FIXED CONFIG")
MC_FIXED = mcnemar_matrix({v: predict(v) for v in MAIN})
print(MC_FIXED.to_string(index=False))
print("\nBEST-ON-SET thresholds (optimistic for the tunable variants)")
MC_BEST = mcnemar_matrix({v: SWEEPS[v]["P"][SWEEPS[v]["best"]] for v in MAIN})
print(MC_BEST.to_string(index=False))
print("\nLeave-one-out component ablations vs full hybrid (fixed config)")
full = predict("4 hybrid (repo)")
MC_COMP = pd.DataFrame([{"ablated": v, **dict(zip(["hybrid only right", "ablated only right", "p (exact)"], mcnemar_exact(full == Y, predict(v) == Y)))} for v in LOO_COMP])
print(MC_COMP.round(4).to_string(index=False))

## 11. Error analysis (which cases does each main variant get wrong at the fixed config?)

In [ ]:
wrong = pd.DataFrame({v: predict(v) != Y for v in MAIN}); wrong.insert(0, "type", TYPES); wrong.insert(0, "id", FEAT.id.values)
wrong.insert(2, "truth", np.where(Y, "consistent", "HALLUC"))
wrong["n_variants_wrong"] = wrong[MAIN].sum(1)
cols = ["ged_sym", "ged_asym", "min_span", "nli_templ", "nli_raw"]
err = pd.concat([wrong, FEAT[cols].round(3)], axis=1)
print("(True = that variant is WRONG on the case)")
print(err[err.n_variants_wrong > 0].sort_values(["n_variants_wrong"], ascending=False).to_string(index=False))
print("\nCases that every main variant gets right:", int((wrong.n_variants_wrong == 0).sum()), "/", len(wrong))
DATA_IDX = {r["id"]: r for r in DATA}
for _id in err.loc[wrong["4 hybrid (repo)"].values, "id"].tolist()[:12]:
    r = DATA_IDX[_id]
    print(f"\n[{_id}] type={r['type']} truth={'consistent' if r['label'] else 'HALLUC'}\n  H: {r['hypothesis']}\n  P: {r['premise']}")

## 12. Save results
Writes `ablation_mcpevol_results.json` (config, environment, per-case features/predictions, all tables) to the output directory.

In [ ]:
def _df(d): return json.loads(d.reset_index().to_json(orient="records"))
RESULTS = {
    "what": "Ablation of mcp-hallucination-checker components on MCPEvol-Bench (60 cases). n is tiny: CIs wide; best-on-set numbers are optimistic.",
    "environment": {"python": platform.python_version(), "torch": torch.__version__, "transformers": transformers.__version__,
                    "spacy": spacy.__version__, "spacy_model_used": SPACY_USED, "cuda": torch.cuda.is_available(),
                    "nli_model": entailment.MODEL_NAME, "ner_model": "dslim/bert-base-NER", "package_root": str(PKG_ROOT)},
    "config": {"ged_threshold": GED_THR, "entailment_threshold": ENT_THR, "ged_fast_mode_node_limit": FAST_LIMIT,
               "ged_grid": GED_GRID, "ent_grid": ENT_GRID, "n_boot": N_BOOT, "seed": SEED, "smoke_per_group": SMOKE_PER_GROUP},
    "n_cases": len(DATA),
    "fidelity": {"hybrid_vs_runner_disagreements": int((real != composed).sum()), "variant5_vs_patched_run_verification_disagreements": int((real5 != comp5).sum())},
    "fixed_config": _df(T_FIXED), "per_type_fixed": json.loads(PT_FIXED.to_json(orient="records")),
    "sweep": _df(T_SWEEP), "mcnemar_fixed": json.loads(MC_FIXED.to_json(orient="records")),
    "mcnemar_best_on_set": json.loads(MC_BEST.to_json(orient="records")), "mcnemar_component": json.loads(MC_COMP.to_json(orient="records")),
    "cases": [{**{k: (None if (isinstance(v, float) and np.isnan(v)) else (bool(v) if isinstance(v, (np.bool_, bool)) else v))
                  for k, v in row.items()},
               **{"pred " + v: bool(predict(v)[i]) for v in MAIN + EXTRA}} for i, row in enumerate(FEAT.to_dict("records"))],
}
(OUT_DIR / "ablation_mcpevol_results.json").write_text(json.dumps(RESULTS, indent=2, default=float), encoding="utf-8")
print("saved", OUT_DIR / "ablation_mcpevol_results.json")

## 13. (Optional) Part 2 - GED+NLI vs NLI-only on a stratified AggreFact sample
Disabled by default (`ABLATE_AGGREFACT_SAMPLE = False`). Needs the LLM-AggreFact `test-*.parquet` attached as a Kaggle dataset (columns `doc`, `claim`, `label`, optional `dataset`).

Design / caveats:
* seeded sample of `AGG_N` rows stratified by (source dataset, label);
* **NLI-only**: whole claim vs the **full** document, repo windowing (180/40 words, max over windows), repo hypothesis template, threshold = repo entailment threshold;
* **GED**: graphs are built from the claim and from the document **truncated to `AGG_GRAPH_DOC_WORDS` words** (spaCy/NER cost and the 512-token limit make full-document graphs impractical), so GED can only see the beginning of the document and is *approximate*; when claim+doc graph nodes exceed the fast-mode limit (80) the repo switches to a degree-sequence approximation, which is crude. Fast-mode rows are counted and reported;
* **hybrid** = GED <= ged threshold AND no argument span below the entailment threshold (spans of the claim scored against the **full** document, windowed); **NLI+GED (AND)** = whole-claim NLI >= threshold AND GED <= threshold;
* every record's graph/GED step runs under a wall-clock timeout (`AGG_TIMEOUT_S`, SIGALRM, main thread, Linux); timed-out / failed rows are excluded from the GED-based variants and counted, and the paired comparison is done on rows where every variant produced a score;
* GED was designed for short claims vs short evidence, and the distance between a short claim and a long document is dominated by document size - expect GED-based variants to be weak here. Not tuned; fixed thresholds only (AUROC of the raw scores is also given).

In [ ]:
import signal
class _Timeout(Exception): pass

@contextlib.contextmanager
def time_limit(sec):
    if sec and hasattr(signal, "SIGALRM") and threading.current_thread() is threading.main_thread():
        def _h(signum, frame): raise _Timeout()
        old = signal.signal(signal.SIGALRM, _h); signal.setitimer(signal.ITIMER_REAL, sec)
        try: yield
        finally:
            signal.setitimer(signal.ITIMER_REAL, 0); signal.signal(signal.SIGALRM, old)
    else:
        yield

def stratified_sample(df, n, seed, cols):
    rng = np.random.default_rng(seed)
    keys = df[cols].astype(str).agg("|".join, axis=1).values
    groups = {k: np.where(keys == k)[0] for k in np.unique(keys)}
    raw = {k: n * len(ix) / len(df) for k, ix in groups.items()}
    alloc = {k: min(len(groups[k]), int(math.floor(v))) for k, v in raw.items()}
    for k in sorted(raw, key=lambda k: raw[k] - math.floor(raw[k]), reverse=True):     # largest remainder
        if sum(alloc.values()) >= n: break
        if alloc[k] < len(groups[k]): alloc[k] += 1
    pick = np.concatenate([rng.choice(groups[k], size=a, replace=False) for k, a in alloc.items() if a > 0])
    return df.iloc[np.sort(pick)].reset_index(drop=True)

def auroc(y, s):
    from scipy.stats import rankdata
    y = np.asarray(y, bool); s = np.asarray(s, float); n1, n0 = int(y.sum()), int((~y).sum())
    if n1 == 0 or n0 == 0: return float("nan")
    return float((rankdata(s)[y].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))

def run_aggrefact_part():
    from tqdm.auto import tqdm
    files = sorted(glob.glob(f"{AGG_DIR}/test-*.parquet")) if AGG_DIR else sorted(glob.glob("/kaggle/input/**/test-*.parquet", recursive=True))
    if not files:
        print("No test-*.parquet found (attach the LLM-AggreFact dataset or set ABL_AGG_DIR). Skipping Part 2."); return None
    df = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
    assert {"doc", "claim", "label"} <= set(df.columns), list(df.columns)
    df["label"] = df["label"].astype(int)
    strat = ["dataset", "label"] if "dataset" in df.columns else ["label"]
    S = stratified_sample(df, AGG_N, AGG_SEED, strat)
    print(f"AggreFact sample: {len(S)} of {len(df)} rows | supported fraction {S.label.mean():.3f} | strata {strat}")
    rows = []
    for i, r in enumerate(tqdm(S.itertuples(index=False), total=len(S))):
        doc, claim = r.doc, r.claim
        rec = dict(i=i, label=bool(r.label), doc_words=len(doc.split()), nli=np.nan, ged=np.nan, ged_asym=np.nan, min_span=np.nan,
                   fast_mode=None, status="ok")
        try:
            rec["nli"] = entailment.score_spans(doc, [claim])[0][1]               # full doc, windowed
        except Exception as ex:
            rec["status"] = f"nli_error:{type(ex).__name__}"
        try:
            dtrunc = " ".join(doc.split()[:AGG_GRAPH_DOC_WORDS])
            with time_limit(AGG_TIMEOUT_S):
                Gh, Gd = build_semantic_graph(claim), build_semantic_graph(dtrunc)
                rec["ged"] = compute_ged(Gh, Gd)[1]
                rec["ged_asym"] = compute_ged_asymmetric(Gh, Gd)[1]
                rec["fast_mode"] = bool(Gh.number_of_nodes() + Gd.number_of_nodes() > FAST_LIMIT)
                spans = _extract_argument_spans(extract_frames(claim))
            sp = entailment.score_spans(doc, spans) if spans else []              # full doc, windowed
            rec["min_span"] = min((s for _, s in sp), default=1.0)
        except _Timeout:
            rec["status"] = "ged_timeout"
        except Exception as ex:
            rec["status"] = f"ged_error:{type(ex).__name__}"
        rows.append(rec)
        if (i + 1) % 50 == 0:
            pd.DataFrame(rows).to_json(OUT_DIR / "ablation_aggrefact_partial.json", orient="records")
    R = pd.DataFrame(rows)
    print("status counts:", R.status.value_counts().to_dict(), "| fast-mode rows:", int((R.fast_mode == True).sum()))
    ok = R[(R.status == "ok")].reset_index(drop=True)          # rows where every variant has a score (paired comparison)
    y = ok.label.values
    V = {"NLI-only (whole claim, full doc)": ok.nli.values >= ENT_THR,
         "GED-only (truncated-doc graph)": ok.ged.values <= GED_THR,
         "hybrid: GED & span-NLI gate": (ok.ged.values <= GED_THR) & (ok.min_span.values >= ENT_THR),
         "NLI-whole AND GED": (ok.nli.values >= ENT_THR) & (ok.ged.values <= GED_THR),
         "[EXP] NLI-whole AND asym-GED": (ok.nli.values >= ENT_THR) & (ok.ged_asym.values <= GED_THR)}
    tab = pd.DataFrame({k: {**{m: metrics(y, p)[m] for m in ("acc", "bal_acc", "prec_h", "rec_h", "f1_h")}, "acc 95% CI": fmt_ci(bootstrap_ci(y, p)["acc_ci"])} for k, p in V.items()}).T
    print(f"\nPaired rows: {len(ok)} | thresholds: ged={GED_THR}, entailment={ENT_THR}. Positive class for P/R/F1 = unsupported (hallucinated).")
    print(tab.round(3).to_string())
    print("\nAUROC of the continuous scores (supported = positive): NLI whole-claim:", round(auroc(y, ok.nli.values), 3),
          "| 1-GED:", round(auroc(y, -ok.ged.values), 3), "| 1-asymGED:", round(auroc(y, -ok.ged_asym.values), 3))
    base = "NLI-only (whole claim, full doc)"
    mc = [{"A": base, "B": k, **dict(zip(["only A right", "only B right", "p (exact)"], mcnemar_exact(V[base] == y, p == y)))} for k, p in V.items() if k != base]
    print("\nMcNemar vs NLI-only:"); print(pd.DataFrame(mc).round(4).to_string(index=False))
    R.to_json(OUT_DIR / "ablation_aggrefact_rows.json", orient="records")
    return dict(table=tab, rows=R)

AGG_RESULT = run_aggrefact_part() if ABLATE_AGGREFACT_SAMPLE else None
if not ABLATE_AGGREFACT_SAMPLE:
    print("Part 2 skipped (ABLATE_AGGREFACT_SAMPLE = False).")

## 14. How to read this
* Compare rows within a table, but remember n = 60: look at the McNemar table before claiming that one variant is better.
* The routed hybrid can only differ from `schema-only` on the 35 text cases; the per-type table shows where the text components help or hurt.
* If `fixed` and `best-on-set` differ a lot while `LOO-CV` stays near `fixed`, the gain from tuning is probably over-fitting the 60 cases.
* Variant 5 is a research sketch; it is not wired into the server and its threshold scale is different.